In [0]:
from pyspark.sql import functions as sf

df = spark.table("medical_data.bronze.payers") \
    .select("id", "name", "address", "city", "state_headquartered", "zip")

df_clean = df \
    .fillna({
        "name": "NA",
        "address": "NA",
        "city": "NA",
        "state_headquartered": "NA",
        "zip": "NA"
    }) \
    .withColumn("zip", sf.regexp_replace(sf.col("zip"), r"\D", "")) \
    .withColumn("name", sf.lower(sf.trim(sf.col("name")))) \
    .withColumn("city", sf.lower(sf.trim(sf.col("city")))) \
    .withColumn("state_headquartered", sf.upper(sf.trim(sf.col("state_headquartered")))) \
    .dropDuplicates(["id"]) \
    .filter(sf.col("id").isNotNull())

df_clean.write.mode("overwrite").saveAsTable("medical_data.silver.payers_s")